# Measure RF-DETR Latency: The Fastest Configuration on Your Hardware

Time every current pretrained RF-DETR checkpoint at batch 1 through `RFDETRInference.predict`, try each runtime configuration the selected hardware supports and keep the fastest one per model. The accuracy cookbook scores fixed precisions (PyTorch FP32, TensorRT FP16 and INT8) to show what each precision costs in AP; this notebook answers a different question, how fast can this model go on this hardware, so the winning configuration can differ per model and per device.

- The measurement is end to end: preprocessing, the forward pass, postprocessing and, for segmentation, decoding the masks on the host. It is not the model-only latency of the Benchmarks page.
- Every call uses the default `threshold=0.5` on `N_IMAGES` COCO val2017 images, each predicted once in the same order for every configuration. Latency is reported as the median call time with its 95th percentile, because one slow call (OS jitter, throttling) moves a mean but not a median.
- Nothing is detected or substituted for you. You choose `HARDWARE`; a missing device, package or runtime stops the notebook, and every predictor is checked against the runtime its configuration names before it is timed.
- This notebook does not score accuracy. Static INT8 loses most AP on the PE-Core-T models (Atto, Femto and Pico), so those never try INT8; see the AP columns of the Run pages.

## Choose the hardware and install

`HARDWARE` selects the runtimes to install and the configurations to try: `cuda` (NVIDIA GPU: PyTorch with a CUDA graph and TensorRT engines), `apple` (Apple Silicon: PyTorch on MPS and CoreML) or `cpu` (ONNX Runtime and OpenVINO on the CPU, plus PyTorch on the CPU). Set it before installing and restart the kernel after the install, so the recorded package versions are the ones in use; then run the cells again from the top.

In [ ]:
HARDWARE = "cuda"
INSTALL_EXTRAS = {"cuda": "plus,tensorrt", "apple": "plus,coreml", "cpu": "plus,onnx,openvino"}
# OpenVINO INT8 needs nncf, and the CPU thread count comes from psutil.
INSTALL_PACKAGES = {"cuda": "", "apple": "", "cpu": "nncf psutil"}
assert HARDWARE in INSTALL_EXTRAS, f"HARDWARE must be one of {tuple(INSTALL_EXTRAS)}, got {HARDWARE!r}."

In [ ]:
!pip install -U "rfdetr[{INSTALL_EXTRAS[HARDWARE]}]" {INSTALL_PACKAGES[HARDWARE]}

Keep imports together so every dependency of the measurement path is visible in one place.

In [ ]:
import importlib.metadata
import itertools
import json
import re
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from urllib.parse import quote

import pandas as pd
import torch
from IPython.display import display
from PIL import Image
from tqdm.auto import tqdm

import rfdetr
from rfdetr import RFDETR, RFDETRInference
from rfdetr.export._benchmark import (
    _UNKNOWN_CPU_BRAND,
    BenchmarkResult,
    _download,
    cpu_brand,
    export_cached,
    fetch_coco_val2017,
    measure_latency,
)

# The TensorRT FP16 export clamps float32 weights below 1e-7 in magnitude to ±1e-7 and warns once per weight;
# the clamping is expected and the warnings would bury the output.
warnings.filterwarnings("ignore", message="the float32 number .* will be truncated to", module="onnxconverter_common")
# Tracing the model for ONNX export warns wherever a tensor value steers Python control flow; the export
# fixes those values for the export resolution, which is intended, so the warnings are noise here.
warnings.filterwarnings("ignore", category=torch.jit.TracerWarning)

## Select the checkpoints and configurations

A configuration is one way to run a model, named `<runtime>-<precision>`; a PyTorch name also carries its compile backend, and `-graph` replays a TensorRT engine as a CUDA graph through the `cuda_graph` runtime option. Every export is a static batch-1 artifact.

> **Note:** PML 1.0 covers the Plus detection models (Atto, Femto, Pico, XLarge and 2XLarge): commercial use is included when you deploy on Roboflow, and commercial self-hosting requires the Enterprise add-on. Apache 2.0 covers the package, Nano–Large detection, every Seg size and Keypoint: commercial, closed-source and self-hosted use is included without a license fee or an obligation to open-source your code. Keep `ACCEPT_PLATFORM_MODEL_LICENSE = True` only if you accept the Plus model terms; set it to `False` to time the Apache 2.0 models alone.

In [ ]:
MODELS = (
    "RFDETRNano",
    "RFDETRSmall",
    "RFDETRMedium",
    "RFDETRLarge",
    "RFDETRSegNano",
    "RFDETRSegSmall",
    "RFDETRSegMedium",
    "RFDETRSegLarge",
    "RFDETRSegXLarge",
    "RFDETRSeg2XLarge",
    "RFDETRKeypointPreview",
)
PLUS_MODELS = ("RFDETRAtto", "RFDETRFemto", "RFDETRPico", "RFDETRXLarge", "RFDETR2XLarge")
ACCEPT_PLATFORM_MODEL_LICENSE = True
if ACCEPT_PLATFORM_MODEL_LICENSE:
    MODELS += PLUS_MODELS
# The PE-Core-T models lose 7.0 to 8.7 AP in static TensorRT INT8 (Run pages). No INT8 configuration is timed for
# them on any hardware, because a faster INT8 artifact is not a usable answer when its accuracy was not checked.
INT8_EXCLUDED = ("RFDETRAtto", "RFDETRFemto", "RFDETRPico")
HARDWARE_CONFIGS = {
    "cuda": (
        "pytorch-cudagraph-fp16",
        "tensorrt-fp16",
        "tensorrt-fp16-graph",
        "tensorrt-int8",
        "tensorrt-int8-graph",
    ),
    "apple": (
        "pytorch-inductor-fp32",
        "pytorch-inductor-fp16",
        "pytorch-torchscript-fp16",
        "coreml-fp32",
        "coreml-fp16",
        "coreml-gpu-fp32",
        "coreml-gpu-fp16",
    ),
    "cpu": (
        "pytorch-torchscript-fp32",
        "onnx-fp32",
        "onnx-int8",
        "openvino-fp32",
        "openvino-fp16",
        "openvino-int8",
    ),
}
CONFIGS = HARDWARE_CONFIGS[HARDWARE]
COCO_ROOT = Path("coco")
WORK_DIR = Path("coco_latency")
RESULTS_PATH = WORK_DIR / "results.jsonl"
N_IMAGES = 500
N_WARMUP = 20
N_CALIB = 128

Reject missing license acceptance and a hardware choice the machine cannot satisfy before downloading data or weights. Model classes resolve through `rfdetr` later, so Plus imports stay lazy and an unknown name fails there.

In [ ]:
assert MODELS, "Select at least one model."
assert ACCEPT_PLATFORM_MODEL_LICENSE or not set(MODELS) & set(PLUS_MODELS), (
    "Plus models require ACCEPT_PLATFORM_MODEL_LICENSE = True after accepting PML 1.0."
)
assert HARDWARE != "cuda" or torch.cuda.is_available(), "HARDWARE = 'cuda' needs a CUDA GPU, and none is available."
assert HARDWARE != "apple" or torch.backends.mps.is_available(), "HARDWARE = 'apple' needs Apple Silicon with MPS."

On the CPU the thread count decides the result, so pin it: PyTorch and OpenVINO use the physical core count, which is also what ONNX Runtime uses by default and cannot be set through `RFDETRInference`. A host that does not report its physical cores stops the notebook rather than guessing.

In [ ]:
if HARDWARE == "cpu":
    import psutil

    CPU_THREADS = psutil.cpu_count(logical=False)
    assert CPU_THREADS, "psutil cannot report the physical core count of this host."
    torch.set_num_threads(CPU_THREADS)
    OPENVINO_OPTIONS = {"config": {"INFERENCE_NUM_THREADS": CPU_THREADS}}
else:
    CPU_THREADS = None
    OPENVINO_OPTIONS = None
print(f"CPU threads: {CPU_THREADS}")

## Prepare the images

Fetch the annotation archive and the `N_IMAGES` val2017 images selected by a fixed seed; images already on disk, for example from the accuracy cookbook, are reused. The annotation archive also carries the train2017 file list the INT8 calibration needs.

In [ ]:
subset = fetch_coco_val2017(COCO_ROOT, n_images=N_IMAGES)
images = [Image.open(subset.images_dir / f"{image_id:012d}.jpg").convert("RGB") for image_id in subset.image_ids]
print(f"Timing with {len(images)} val2017 images, each predicted once in a fixed order.")

## Record provenance and resume completed runs

Record the hardware, the device name, the installed Git commit when available and the version of every package the selected runtimes need once; a package that is not installed stops the notebook. Resume identifies a row by model, configuration, hardware, device, thread count, image count, RF-DETR version and commit, so a rerun continues where it stopped and a newer commit or another machine measures again.

In [ ]:
cpu = cpu_brand()
assert HARDWARE == "cuda" or cpu != _UNKNOWN_CPU_BRAND, (
    "The host does not report a CPU model, so the device cannot be named."
)

In [ ]:
installed_versions = {
    dist.metadata["Name"].lower().replace("_", "-"): dist.version for dist in importlib.metadata.distributions()
}
RUNTIME_PACKAGES = {
    "cuda": ("tensorrt",),
    "apple": ("coremltools",),
    "cpu": ("onnxruntime", "openvino", "nncf", "psutil"),
}
direct_url = json.loads(importlib.metadata.distribution("rfdetr").read_text("direct_url.json") or "{}")
device = {
    "cuda": lambda: f"cuda: {torch.cuda.get_device_name()}",
    "apple": lambda: f"apple: {cpu}",
    "cpu": lambda: f"cpu: {cpu}",
}[HARDWARE]()
provenance = {
    "hardware": HARDWARE,
    "device": device,
    "cpu_threads": CPU_THREADS,
    "n_images": N_IMAGES,
    "rfdetr_version": importlib.metadata.version("rfdetr"),
    "commit": direct_url.get("vcs_info", {}).get("commit_id"),
    "torch_version": torch.__version__,
    "rfdetr_plus_version": installed_versions["rfdetr-plus"],
    **{f"{package.replace('-', '_')}_version": installed_versions[package] for package in RUNTIME_PACKAGES[HARDWARE]},
}
print(json.dumps(provenance, indent=2))

An exported artifact runs only on the hardware and runtime versions that built it. Cache artifacts under both, in a directory per model and configuration. For TensorRT this layout matches the accuracy cookbook, so setting `WORK_DIR` to its directory reuses the engines it built. Use a fresh `WORK_DIR` after changing calibration, weights or other software, because the path records only the device and the first runtime package's version.

In [ ]:
artifact_key = f"{device} {RUNTIME_PACKAGES[HARDWARE][0]} {provenance[RUNTIME_PACKAGES[HARDWARE][0] + '_version']}"
ENGINE_DIR = WORK_DIR / "engines" / re.sub(r"[^a-z0-9.]+", "-", artifact_key.lower()).strip("-")
print(f"Artifact cache: {ENGINE_DIR}")

In [ ]:
RESUME_FIELDS = ("model", "config", "hardware", "device", "cpu_threads", "n_images", "rfdetr_version", "commit")


def resume_key(row: dict[str, Any]) -> tuple[Any, ...]:
    """Identify a measurement by its model, configuration and measuring context."""
    return tuple(row[field] for field in RESUME_FIELDS)

In [ ]:
WORK_DIR.mkdir(parents=True, exist_ok=True)
rows = (
    [json.loads(line) for line in RESULTS_PATH.read_text().splitlines() if line.strip()]
    if RESULTS_PATH.exists()
    else []
)
completed = {resume_key(row) for row in rows}
print(f"Loaded {len(rows)} completed measurements from {RESULTS_PATH}.")

## Prepare held-out INT8 calibration

Static INT8 uses the first `N_CALIB` train2017 images sorted by filename, the same set as the accuracy cookbook, disjoint from val2017. Existing files are reused; hardware without an INT8 configuration skips the download.

In [ ]:
CALIBRATION_DIR = WORK_DIR / "calibration"
if any("int8" in config for config in CONFIGS):
    train_images = json.loads((COCO_ROOT / "annotations" / "instances_train2017.json").read_text())["images"]
    calibration_names = sorted(record["file_name"] for record in train_images)[:N_CALIB]
    del train_images
    for name in calibration_names:
        destination = CALIBRATION_DIR / name
        if not destination.exists():
            _download("http://images.cocodataset.org/train2017/" + quote(name), destination)
    print(f"Calibration: {N_CALIB} train2017 images in {CALIBRATION_DIR}.")
else:
    print("No INT8 configuration for this hardware; calibration images are not needed.")

## Build predictors

`EXPORTS` lists, per exported configuration, the artifact pattern and the export keywords. `export_cached` reuses a cached artifact or exports one, and refuses an ambiguous cache or an export that does not match its pattern. The `-graph` variant of a TensorRT precision shares its engine. The `coreml-gpu` configurations share the Core ML package of the same precision and load it with `device="cpu_and_gpu"`, while `coreml` takes the default `device="auto"`, which lets Core ML put work on the Neural Engine.

In [ ]:
INT8 = {"quantization": "int8", "calibration_data": str(CALIBRATION_DIR), "max_images": N_CALIB}
EXPORTS = {
    "tensorrt-fp16": ("*.trt", {"format": "tensorrt", "fp16": True}),
    "tensorrt-int8": ("*_int8.trt", {"format": "tensorrt", **INT8}),
    "onnx-fp32": ("*.onnx", {"format": "onnx"}),
    "onnx-int8": ("*.onnx", {"format": "onnx", **INT8}),
    "openvino-fp32": ("*.xml", {"format": "openvino", "openvino_precision": "float32"}),
    "openvino-fp16": ("*.xml", {"format": "openvino", "openvino_precision": "float16"}),
    "openvino-int8": ("*.xml", {"format": "openvino", **INT8}),
    "coreml-fp32": ("*.mlpackage", {"format": "coreml"}),
    "coreml-fp16": ("*.mlpackage", {"format": "coreml", "coreml_precision": "float16"}),
}

`build_predictor` builds one predictor and then checks it. PyTorch names the compile backend and dtype explicitly instead of taking the device default, and `runtime_info` must report the runtime and device the configuration names; any difference stops the run, so a configuration can never be timed as a different one.

In [ ]:
DTYPES = {"fp32": torch.float32, "fp16": torch.float16}
#: What `runtime_info` must report: (backend, device prefix); a device prefix of None is not checked.
PYTORCH_DEVICE = {"cuda": "cuda", "apple": "mps", "cpu": "cpu"}[HARDWARE]
EXPECTED_DEVICE = {"onnx": "CPUExecutionProvider", "openvino": "CPU", "coreml": "auto"}
#: Configurations that load the artifact of another configuration with explicit Core ML compute units, as (artifact, device).
CORE_ML_UNITS = {
    "coreml-gpu-fp32": ("coreml-fp32", "cpu_and_gpu"),
    "coreml-gpu-fp16": ("coreml-fp16", "cpu_and_gpu"),
}


def build_predictor(model: RFDETR, model_name: str, config: str) -> RFDETRInference:
    """Build the predictor of one configuration and verify the runtime it reports.

    Args:
        model: Freshly constructed pretrained model.
        model_name: Checkpoint label for the cache directory.
        config: One entry of ``CONFIGS``.

    Returns:
        A public predictor owning its runtime.
    """
    backend = config.split("-")[0]
    if backend == "pytorch":
        _, compile_backend, precision = config.split("-")
        predictor = model.inference(compile_backend=compile_backend, dtype=DTYPES[precision])
        expected_device = PYTORCH_DEVICE
    else:
        if config.endswith("-graph"):
            options = {"cuda_graph": True}
        else:
            options = OPENVINO_OPTIONS if backend == "openvino" else None
        exported, core_ml_device = CORE_ML_UNITS.get(config, (config.removesuffix("-graph"), None))
        pattern, export_kwargs = EXPORTS[exported]
        artifact = export_cached(
            model, ENGINE_DIR / model_name / exported, pattern, batch_size=1, verbose=False, **export_kwargs
        )
        load_kwargs = {"device": "cpu"} if HARDWARE == "cpu" else {}
        if core_ml_device is not None:
            load_kwargs = {"device": core_ml_device}
        predictor = RFDETRInference(artifact, runtime_options=options, **load_kwargs)
        expected_device = core_ml_device or EXPECTED_DEVICE.get(backend)
    info = predictor.runtime_info
    assert info["backend"] == backend, f"{config} loaded the {info['backend']} runtime, expected {backend}."
    assert expected_device is None or str(info["device"]).startswith(expected_device), (
        f"{config} runs on {info['device']}, expected {expected_device}."
    )
    return predictor

`measure_predictor` times one predictor: it predicts each image once, so a configuration never sees a single easy frame, and returns every call's time in milliseconds. CUDA events time the call on a GPU and wall-clock time any other hardware; `predict` returns host results, so either one covers the whole call.

In [ ]:
def measure_predictor(predictor: RFDETRInference, config: str) -> BenchmarkResult:
    """Time batch-1 end-to-end prediction of one predictor, one call per image after ``N_WARMUP`` untimed ones.

    Args:
        predictor: Public predictor to time.
        config: Configuration label for the result row.

    Returns:
        The timed calls, with their mean, standard deviation, median and 95th percentile.
    """
    cycle = itertools.count()
    return measure_latency(
        lambda: predictor.predict(images[next(cycle) % len(images)], include_source_image=False),
        label=config,
        device="cuda" if HARDWARE == "cuda" else "cpu",
        warmup=N_WARMUP,
        runs=len(images),
    )

## Measure and save each run

List the model and configuration pairs still to measure; INT8 is skipped for `INT8_EXCLUDED`. Each run times one call per image after `N_WARMUP` untimed ones and appends its row at once so an interruption keeps every finished measurement. A configuration that fails to build, load or match its runtime stops the loop; fix the cause and rerun to continue from the saved rows.

In [ ]:
pairs = [(m, c) for m in MODELS for c in CONFIGS if not ("int8" in c and m in INT8_EXCLUDED)]
pending = [(m, c) for m, c in pairs if resume_key({**provenance, "model": m, "config": c}) not in completed]
print(f"{len(pending)} of {len(pairs)} runs to measure; {len(pairs) - len(pending)} already measured.")

In [ ]:
loaded_name, model = None, None
progress = tqdm(pending, desc="Runs", unit="run")
for model_name, config in progress:
    progress.set_postfix_str(f"{model_name} / {config}")
    if model_name != loaded_name:
        model = None
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        # The license keyword exists only on the Plus constructors; the CPU runs pin the model to the CPU.
        model_kwargs = {"accept_platform_model_license": True} if model_name in PLUS_MODELS else {}
        if HARDWARE == "cpu":
            model_kwargs["device"] = "cpu"
        model = getattr(rfdetr, model_name)(**model_kwargs)
        loaded_name = model_name
    predictor = build_predictor(model, model_name, config)
    result = measure_predictor(predictor, config)
    info = predictor.runtime_info
    row = {
        **provenance,
        "model": model_name,
        "config": config,
        "task": info["task"],
        "backend": info["backend"],
        "runtime_device": str(info["device"]),
        "measured_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "median_ms": result.median_ms,
        "p95_ms": result.p95_ms,
        "mean_ms": result.mean_ms,
        "std_ms": result.std_ms,
    }
    with RESULTS_PATH.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, allow_nan=False) + "\n")
    completed.add(resume_key(row))
    rows.append(row)
    progress.write(f"{model_name} / {config}: median {result.median_ms:.2f} ms, p95 {result.p95_ms:.2f} ms")
    del predictor
model = None
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Fastest configuration per model

`rows` holds the saved measurements loaded at the start plus this session's. Keep those of this hardware, device, version and commit for the selected models and configurations, and pick the lowest median latency per model. The 95th percentile shows how slow the occasional call is, and the mean and standard deviation stay in `results.jsonl`. The last column counts the configurations that finished, so a winner chosen from an incomplete sweep is visible.

In [ ]:
wanted = {resume_key({**provenance, "model": m, "config": c}) for m, c in pairs}
current = pd.DataFrame([row for row in rows if resume_key(row) in wanted])
for field, value in provenance.items():
    print(f"{field}: {value}")

In [ ]:
# A configuration name is the engine and the precision joined by the last dash, and the tables read better with them in separate columns.
engine_precision = current["config"].str.rsplit("-", n=1, expand=True)
current = current.assign(engine=engine_precision[0], precision=engine_precision[1])

In [ ]:
order = {name: index for index, name in enumerate(MODELS)}
best = current.loc[current.groupby("model")["median_ms"].idxmin()]
counts = current.groupby("model")["config"].nunique()
expected = {m: sum(1 for pair in pairs if pair[0] == m) for m in MODELS}
table = best.assign(
    configs_done=[f"{counts[m]}/{expected[m]}" for m in best["model"]],
    ms=best["median_ms"].round(2),
    p95=best["p95_ms"].round(2),
)
table = table.sort_values("model", key=lambda column: column.map(order))
display(table[["model", "task", "engine", "precision", "ms", "p95", "configs_done"]].reset_index(drop=True))

## Every configuration

The full sweep, so you can see how far the winner is ahead. Each cell is median ms (p95): the median latency in milliseconds, with the 95th percentile of the same calls, also in milliseconds, in brackets, with one column per engine and precision.

In [ ]:
full = current.assign(
    cell=current["median_ms"].map("{:.2f}".format) + " (" + current["p95_ms"].map("{:.2f}".format) + ")"
)
display(
    full.pivot(index="model", columns=["engine", "precision"], values="cell").loc[
        [m for m in MODELS if m in set(full["model"])]
    ]
)

## Next steps

Use [the accuracy cookbook](../coco-accuracy/) to check what a precision costs in AP before choosing its configuration. The numbers depend on the hardware, the runtime versions and the host CPU, because the host does the preprocessing and postprocessing, so state them next to any copied figure. The model-only latency of the [Benchmarks page](https://rfdetr.roboflow.com/latest/learn/benchmarks/) is a different measurement.